# 🌦️ M2_02: Weather Data API - SOLUTIONS

This notebook contains complete solutions for M2_02_weather_data_api.ipynb

**Instructions**: Use this notebook to check your work after attempting the tasks in the main notebook.

## Setup

Run the same setup as the main notebook.

In [ ]:
# Import libraries
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from pathlib import Path

# Configuration
BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'
AMSTERDAM_LAT = 52.3676
AMSTERDAM_LON = 4.9041
START_DATE = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
END_DATE = datetime.now().strftime('%Y-%m-%d')
WEATHER_VARS = [
    'temperature_2m',
    'relativehumidity_2m',
    'precipitation',
    'rain',
    'snowfall',
    'windspeed_10m',
    'winddirection_10m',
    'cloudcover'
]

print("✅ Setup complete")

---

## Task 1: Fetch Weather Data

### Solution

In [ ]:
def fetch_weather_data(latitude, longitude, start_date, end_date, 
                       variables, timezone='Europe/Amsterdam', timeout=30):
    """
    Fetch historical weather data from Open-Meteo API.
    
    Parameters:
    -----------
    latitude : float
        Latitude coordinate
    longitude : float
        Longitude coordinate
    start_date : str or date
        Start date (YYYY-MM-DD)
    end_date : str or date
        End date (YYYY-MM-DD)
    variables : list
        List of weather variable names to fetch
    timezone : str
        Timezone for timestamps (default: 'Europe/Amsterdam')
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None otherwise
    """
    # Build query parameters
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date if isinstance(start_date, str) else start_date.strftime('%Y-%m-%d'),
        'end_date': end_date if isinstance(end_date, str) else end_date.strftime('%Y-%m-%d'),
        'hourly': ','.join(variables),
        'timezone': timezone
    }
    
    print(f"📡 Fetching weather data from Open-Meteo API...")
    print(f"📍 Location: ({latitude}, {longitude})")
    print(f"📅 Date range: {params['start_date']} to {params['end_date']}")
    
    try:
        # Make API request
        response = requests.get(BASE_URL, params=params, timeout=timeout)
        
        # Check status
        if response.status_code == 200:
            print(f"✅ Success! Status code: {response.status_code}")
            data = response.json()
            
            # Print summary
            if 'hourly' in data:
                print(f"📊 Received {len(data['hourly']['time'])} hourly records")
                print(f"🌦️ Variables: {', '.join(data['hourly'].keys())}")
            
            return data
        else:
            print(f"❌ Error: HTTP {response.status_code}")
            print(f"Message: {response.text}")
            return None
            
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout: Request took longer than {timeout} seconds")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching weather data: {e}")
        return None


# Test the function
print("=" * 70)
weather_data = fetch_weather_data(
    AMSTERDAM_LAT, 
    AMSTERDAM_LON,
    START_DATE,
    END_DATE,
    WEATHER_VARS
)
print("=" * 70)

### Key Points

- Build params dictionary with all required parameters
- Join variables list into comma-separated string
- Use try/except for robust error handling
- Check status_code before parsing JSON
- Provide informative error messages

### Common Mistakes

❌ Forgetting to join the variables list: `'hourly': variables` (wrong)  
✅ Correct: `'hourly': ','.join(variables)`

❌ Not checking status code before calling `.json()`  
✅ Always check `response.status_code == 200` first

---

## Task 2: Convert to DataFrame

### Solution

In [ ]:
# Extract hourly data
hourly_data = weather_data['hourly']

# Convert to DataFrame
df_weather = pd.DataFrame(hourly_data)

# Parse timestamps
df_weather['time'] = pd.to_datetime(df_weather['time'])

# Rename for clarity
df_weather = df_weather.rename(columns={'time': 'timestamp'})

# Add metadata
df_weather['latitude'] = weather_data['latitude']
df_weather['longitude'] = weather_data['longitude']
df_weather['elevation_m'] = weather_data['elevation']

print(f"✅ Created weather DataFrame with {len(df_weather)} rows and {len(df_weather.columns)} columns")
print(f"\n📊 First few rows:")
display(df_weather.head())

print("\n" + "=" * 60)
print("📋 Dataset Info:")
print("=" * 60)
df_weather.info()

### Key Points

- Extract the 'hourly' dictionary from the JSON response
- `pd.DataFrame()` can directly convert a dictionary
- Always parse datetime columns with `pd.to_datetime()`
- Add metadata columns for context (lat, lon, elevation)

### Alternative Approaches

You could also use `df_weather['timestamp'] = pd.to_datetime(df_weather['time'])` and keep both columns.

---

## Task 3: Bikeability Score

### Solution

In [ ]:
def calculate_bikeability_score(row):
    """
    Calculate a bikeability score (0-1) based on weather conditions.
    
    Factors:
    - Optimal temp: 15-25°C (score = 1.0)
    - Precipitation: 0mm (score = 1.0), reduces with rain
    - Wind speed: <10 km/h (score = 1.0), reduces with wind
    
    Returns:
    --------
    float : Score between 0 and 1
    """
    score = 1.0
    
    # Temperature factor
    temp = row['temperature_2m']
    if temp < 0:
        score *= 0.2  # Very cold
    elif temp < 10:
        score *= 0.5  # Cold
    elif temp < 15:
        score *= 0.8  # Cool
    elif temp <= 25:
        score *= 1.0  # Optimal
    elif temp < 30:
        score *= 0.8  # Warm
    else:
        score *= 0.5  # Hot
    
    # Precipitation factor
    precip = row['precipitation']
    if precip > 0:
        score *= max(0.3, 1.0 - precip * 0.2)  # Reduce score with rain
    
    # Wind speed factor
    wind = row['windspeed_10m']
    if wind > 30:
        score *= 0.3  # Very windy
    elif wind > 20:
        score *= 0.6  # Windy
    elif wind > 10:
        score *= 0.8  # Breezy
    
    return max(0, min(1, score))  # Clamp to [0, 1]


# Apply the function
df_weather['bikeability_score'] = df_weather.apply(calculate_bikeability_score, axis=1)

print("📊 Bikeability Score Statistics:")
print(f"Mean: {df_weather['bikeability_score'].mean():.2f}")
print(f"Min: {df_weather['bikeability_score'].min():.2f}")
print(f"Max: {df_weather['bikeability_score'].max():.2f}")

# Find best and worst times
best_idx = df_weather['bikeability_score'].idxmax()
worst_idx = df_weather['bikeability_score'].idxmin()

print(f"\n✅ Best biking weather:")
print(f"   Time: {df_weather.loc[best_idx, 'timestamp']}")
print(f"   Temp: {df_weather.loc[best_idx, 'temperature_2m']:.1f}°C")
print(f"   Rain: {df_weather.loc[best_idx, 'precipitation']:.1f}mm")
print(f"   Wind: {df_weather.loc[best_idx, 'windspeed_10m']:.1f}km/h")
print(f"   Score: {df_weather.loc[best_idx, 'bikeability_score']:.2f}")

print(f"\n❌ Worst biking weather:")
print(f"   Time: {df_weather.loc[worst_idx, 'timestamp']}")
print(f"   Temp: {df_weather.loc[worst_idx, 'temperature_2m']:.1f}°C")
print(f"   Rain: {df_weather.loc[worst_idx, 'precipitation']:.1f}mm")
print(f"   Wind: {df_weather.loc[worst_idx, 'windspeed_10m']:.1f}km/h")
print(f"   Score: {df_weather.loc[worst_idx, 'bikeability_score']:.2f}")

### Key Points

- Use `.apply(function, axis=1)` to process each row
- Start with score = 1.0 and multiply by factors
- Use realistic thresholds based on domain knowledge
- Clamp the final score to [0, 1] range

### Alternative Approaches

You could use more sophisticated formulas:
- Exponential decay for precipitation
- Gaussian distribution for temperature
- Weighted combination of factors

### Common Mistakes

❌ Forgetting `axis=1` in `.apply()` - processes columns instead of rows  
❌ Not clamping the score - can go outside [0, 1]  
❌ Using additive instead of multiplicative factors - less realistic

---

## Summary

### What We've Accomplished

1. ✅ Fetched historical weather data from Open-Meteo API
2. ✅ Converted JSON to DataFrame with proper datetime parsing
3. ✅ Created a derived feature (bikeability score)
4. ✅ Analyzed weather patterns

### Key Takeaways

- Weather APIs provide rich historical data for analysis
- Domain knowledge is essential for feature engineering
- Always validate data before using in downstream tasks
- Document assumptions in your feature calculations

### Next Steps

- Proceed to M2_03 for data storage best practices
- In M2_04, we'll merge bike and weather data for combined analysis

---

## Task 6.2: Weather Pattern Analysis

### Solution

In [ ]:
# 1. Coldest hour
coldest_idx = df_weather['temperature_2m'].idxmin()
print("❄️ Coldest Hour:")
print(f"   Time: {df_weather.loc[coldest_idx, 'timestamp']}")
print(f"   Temperature: {df_weather.loc[coldest_idx, 'temperature_2m']:.1f}°C")
print(f"   Wind: {df_weather.loc[coldest_idx, 'windspeed_10m']:.1f} km/h")
print(f"   Precipitation: {df_weather.loc[coldest_idx, 'precipitation']:.1f} mm")

# 2. Rainiest hour
rainiest_idx = df_weather['precipitation'].idxmax()
print("\n🌧️ Rainiest Hour:")
print(f"   Time: {df_weather.loc[rainiest_idx, 'timestamp']}")
print(f"   Precipitation: {df_weather.loc[rainiest_idx, 'precipitation']:.1f} mm")
print(f"   Temperature: {df_weather.loc[rainiest_idx, 'temperature_2m']:.1f}°C")
print(f"   Wind: {df_weather.loc[rainiest_idx, 'windspeed_10m']:.1f} km/h")

# 3. Percentage of good biking weather
good_weather_hours = (df_weather['bikeability_score'] > 0.7).sum()
total_hours = len(df_weather)
good_weather_pct = (good_weather_hours / total_hours) * 100
print(f"\n✅ Good Biking Weather (score > 0.7):")
print(f"   {good_weather_hours} hours out of {total_hours} ({good_weather_pct:.1f}%)")

# 4. Longest stretch without rain
no_rain = df_weather['precipitation'] == 0
# Find consecutive groups
groups = (no_rain != no_rain.shift()).cumsum()
no_rain_groups = df_weather[no_rain].groupby(groups[no_rain]).size()
if len(no_rain_groups) > 0:
    longest_stretch = no_rain_groups.max()
    print(f"\n🌤️ Longest Stretch Without Rain:")
    print(f"   {longest_stretch} consecutive hours")
else:
    print(f"\n🌧️ It rained every hour in this period!")

### Key Points

- Use `.idxmin()` and `.idxmax()` to find indices of extreme values
- Use `.loc[index]` to retrieve complete row data
- Boolean filtering with `.sum()` counts True values
- Groupby with cumsum() identifies consecutive sequences

---

## Task 6.3: Weather Comparison Visualization

### Solution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('🌦️ Weather Analysis Dashboard', fontsize=16, fontweight='bold')

# Panel 1: Temperature distribution
axes[0, 0].hist(df_weather['temperature_2m'], bins=30, color='coral', edgecolor='black', alpha=0.7)
temp_mean = df_weather['temperature_2m'].mean()
axes[0, 0].axvline(temp_mean, color='red', linestyle='--', linewidth=2, 
                   label=f'Mean: {temp_mean:.1f}°C')
axes[0, 0].set_xlabel('Temperature (°C)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Temperature Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Panel 2: Precipitation over time
total_rain = df_weather['precipitation'].sum()
axes[0, 1].bar(df_weather['timestamp'], df_weather['precipitation'], 
               color='steelblue', alpha=0.7, width=0.04)
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Precipitation (mm)')
axes[0, 1].set_title(f'Precipitation Over Time (Total: {total_rain:.1f}mm)')
axes[0, 1].grid(True, alpha=0.3)

# Panel 3: Wind vs Temperature scatter (colored by bikeability)
scatter = axes[1, 0].scatter(df_weather['temperature_2m'], df_weather['windspeed_10m'],
                             c=df_weather['bikeability_score'], cmap='RdYlGn',
                             s=30, alpha=0.6, edgecolors='black', linewidth=0.5)
correlation = df_weather['temperature_2m'].corr(df_weather['windspeed_10m'])
axes[1, 0].set_xlabel('Temperature (°C)')
axes[1, 0].set_ylabel('Wind Speed (km/h)')
axes[1, 0].set_title(f'Wind vs Temperature (Correlation: {correlation:.2f})')
axes[1, 0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1, 0], label='Bikeability Score')

# Panel 4: Bikeability score distribution
axes[1, 1].hist(df_weather['bikeability_score'], bins=20, color='green', 
                edgecolor='black', alpha=0.7)
bike_mean = df_weather['bikeability_score'].mean()
axes[1, 1].axvline(bike_mean, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {bike_mean:.2f}')
good_hours_pct = ((df_weather['bikeability_score'] > 0.7).sum() / len(df_weather)) * 100
axes[1, 1].set_xlabel('Bikeability Score')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title(f'Bikeability Distribution ({good_hours_pct:.1f}% "Good" Hours)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Key Points

- Create subplots with `plt.subplots(rows, cols, figsize)`
- Access individual axes with array indexing: `axes[row, col]`
- Use `c=` parameter in scatter for color mapping by values
- `plt.colorbar()` requires the scatter object and axis reference
- Calculate correlation with `.corr()` method

### Tips

- Set `edgecolors='black'` on scatter plots for better visibility
- Use alpha for overlapping points transparency
- Add meaningful titles that include summary statistics
- Grid helps readers interpret values

---

## Task 6.4: Custom Analysis Example

### Solution (Example: Hourly Weather Patterns)

Here's one possible custom analysis - you could do many different things!

In [ ]:
# Example: Analyze weather patterns by hour of day
df_weather['hour'] = df_weather['timestamp'].dt.hour

# Calculate average conditions by hour
hourly_patterns = df_weather.groupby('hour').agg({
    'temperature_2m': 'mean',
    'precipitation': 'sum',
    'windspeed_10m': 'mean',
    'bikeability_score': 'mean'
}).reset_index()

# Create visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Top panel: Temperature and bikeability by hour
ax1_twin = ax1.twinx()
ax1.plot(hourly_patterns['hour'], hourly_patterns['temperature_2m'], 
         'o-', color='coral', linewidth=2, markersize=6, label='Avg Temperature')
ax1_twin.plot(hourly_patterns['hour'], hourly_patterns['bikeability_score'], 
              's-', color='green', linewidth=2, markersize=6, label='Avg Bikeability')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Temperature (°C)', color='coral')
ax1_twin.set_ylabel('Bikeability Score', color='green')
ax1.set_title('Temperature and Bikeability Patterns by Hour of Day')
ax1.tick_params(axis='y', labelcolor='coral')
ax1_twin.tick_params(axis='y', labelcolor='green')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# Bottom panel: Precipitation and wind
ax2_twin = ax2.twinx()
ax2.bar(hourly_patterns['hour'], hourly_patterns['precipitation'], 
        color='steelblue', alpha=0.7, label='Total Precipitation')
ax2_twin.plot(hourly_patterns['hour'], hourly_patterns['windspeed_10m'], 
              'D-', color='purple', linewidth=2, markersize=6, label='Avg Wind Speed')
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Precipitation (mm)', color='steelblue')
ax2_twin.set_ylabel('Wind Speed (km/h)', color='purple')
ax2.set_title('Precipitation and Wind Patterns by Hour of Day')
ax2.tick_params(axis='y', labelcolor='steelblue')
ax2_twin.tick_params(axis='y', labelcolor='purple')
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Findings
print("\n📊 Findings:")
best_hour = hourly_patterns.loc[hourly_patterns['bikeability_score'].idxmax(), 'hour']
worst_hour = hourly_patterns.loc[hourly_patterns['bikeability_score'].idxmin(), 'hour']
print(f"✅ Best hour for biking: {best_hour:02d}:00")
print(f"❌ Worst hour for biking: {worst_hour:02d}:00")
print(f"\nThis analysis reveals diurnal patterns in weather conditions,")
print(f"showing that bikeability varies significantly throughout the day.")

### Other Ideas for Custom AnalysisYou could also explore:- **Weekend vs Weekday patterns**: Add day of week and compare- **Humidity-Precipitation relationship**: Scatter plot with trend line- **Temperature trends**: Linear regression over the 30-day period- **Weather "comfort zones"**: 2D density plot of temp vs humidity- **Extreme events**: Flag and analyze outlier conditionsThe key is to ask an interesting question and use appropriate visualizations!---## Optional Challenges - Guidance### Challenge 7.1: KNMI Weather API Exploration**This is intentionally a research challenge** - no complete solution provided! Here's guidance:**Key Discovery Points:**1. **KNMI Data Structure:**   - KNMI provides station-based data (not lat/lon like Open-Meteo)   - Station 240 = Schiphol (near Amsterdam)   - Data often comes as CSV or text format, not JSON   - Documentation is primarily in Dutch (good practice for international data sources!)2. **API Endpoint Examples:**   - Daily data: `https://www.daggegevens.knmi.nl/klimatologie/daggegevens`   - Hourly data: Look for "uurgegevens" endpoints   - May require form parameters instead of REST API3. **Data Format Differences:**   ```python   # KNMI often uses:   # - T: Temperature in 0.1 degrees C (divide by 10!)   # - RH: Precipitation in 0.1mm   # - DD/FF: Wind direction/speed   # - Comment lines start with #   ```4. **Parsing Strategy:**   ```python   # Read CSV with comments   df_knmi = pd.read_csv(response.text, comment='#', skipinitialspace=True)      # Convert units   df_knmi['temperature_c'] = df_knmi['T'] / 10   ```5. **Comparison Approach:**   - Match timestamps between KNMI and Open-Meteo   - Calculate correlation: `df['temp_om'].corr(df['temp_knmi'])`   - Plot both on same axes to visualize differences   - Calculate mean absolute difference**Expected Findings:**- High correlation (>0.95) but not perfect- KNMI may have slightly different values (different measurement locations/methods)- KNMI data is authoritative for Netherlands- Open-Meteo is easier to use internationally**Reflection Points:**- Which API had better documentation?- Which would you use for a production Dutch bike app? Why?- What challenges did you face adapting to KNMI's format?### Challenge 7.2: Forecast Integration**Key Differences:**- Use `https://api.open-meteo.com/v1/forecast` (not archive)- No `start_date/end_date` - uses `forecast_days=7`- Returns future predictions instead of historical data### Challenge 7.3: Caching System**Structure:**```pythondef fetch_weather_with_cache(lat, lon, start, end, cache_dir='./cache'):    cache_key = f"weather_{lat}_{lon}_{start}_{end}.json"    cache_path = Path(cache_dir) / cache_key        if cache_path.exists():        # Check age, load if recent        pass        # Fetch and save    pass```### Challenge 7.4: Extreme Weather Alerting**Logic:**```pythondef detect_extremes(df):    extremes = []        # Temperature extremes    cold = df[df['temperature_2m'] < 0]    hot = df[df['temperature_2m'] > 35]        # Rain extremes    heavy_rain = df[df['precipitation'] > 10]        # Wind extremes      strong_wind = df[df['windspeed_10m'] > 40]        return extremes```